Impporting file setup

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch

Loading Phase 6 artifacts

In [2]:
phase6_pairs = pd.read_csv("phase6_anchor_positive_pairs.csv")
phase6_queries = pd.read_csv("phase6_queries.csv")
phase6_gold = pd.read_csv("phase6_gold_positive_map.csv")
retrieval_corpus = pd.read_csv("phase6_retrieval_corpus.csv")

print("Anchor-positive pairs:", phase6_pairs.shape)
print("Queries:", phase6_queries.shape)
print("Gold map:", phase6_gold.shape)
print("Retrieval corpus:", retrieval_corpus.shape)

phase6_pairs.head()

Anchor-positive pairs: (2700, 3)
Queries: (2700, 2)
Gold map: (2700, 4)
Retrieval corpus: (5400, 2)


,anchor,positive,pair_type
0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,question_to_answer
1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,question_to_answer
2,What role do community events or festivals pla...,Community events or festivals provide opportun...,question_to_answer
3,How can blood donation centers collaborate wit...,Blood donation centers can partner with transp...,question_to_answer
4,How do blood donation centers ensure the quali...,Blood donation centers ensure the quality and ...,question_to_answer


Loading the current best MPNet retriever

In [3]:
mpnet_model = SentenceTransformer("all-mpnet-base-v2")
print("MPNet model loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MPNet model loaded successfully.


Encoding the retrieval corpus

In [4]:
corpus_texts = retrieval_corpus["text"].tolist()

corpus_embeddings = mpnet_model.encode(
    corpus_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Corpus embeddings created.")
print("Corpus size:", len(corpus_texts))
print("Embedding shape:", corpus_embeddings.shape)

Batches:   0%|          | 0/169 [00:00<?, ?it/s]

Corpus embeddings created.
Corpus size: 5400
Embedding shape: torch.Size([5400, 768])


Encoding the training queries

In [5]:
query_texts = phase6_queries["query"].tolist()

query_embeddings = mpnet_model.encode(
    query_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Query embeddings created.")
print("Number of queries:", len(query_texts))
print("Embedding shape:", query_embeddings.shape)

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Query embeddings created.
Number of queries: 2700
Embedding shape: torch.Size([2700, 768])


Mining top candidate matches for each query

In [6]:
top_k = 10

all_candidate_matches = []

for i, query in enumerate(query_texts):
    sims = util.cos_sim(query_embeddings[i], corpus_embeddings)[0]
    top_results = sims.topk(k=top_k)
    
    indices = top_results.indices.tolist()
    scores = top_results.values.tolist()
    
    for idx, score in zip(indices, scores):
        all_candidate_matches.append({
            "query_id": i,
            "query": query,
            "candidate_text": corpus_texts[idx],
            "candidate_type": retrieval_corpus.iloc[idx]["text_type"],
            "similarity_score": float(score)
        })

candidate_matches_df = pd.DataFrame(all_candidate_matches)

print("Candidate matches mined.")
candidate_matches_df.head(20)

Candidate matches mined.


,query_id,query,candidate_text,candidate_type,similarity_score
0,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,1.000000
1,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.912461
2,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.908266
3,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.894374
4,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.893983
5,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.891884
6,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.887956
7,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.884563
8,0,How can blood donation centers collaborate wit...,How can blood donation centers partner with he...,question,0.882852
9,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.881042


Attaching gold positives

In [7]:
phase7_candidates = candidate_matches_df.merge(
    phase6_gold[["query_id", "positive"]],
    on="query_id",
    how="left"
)

phase7_candidates.head()

,query_id,query,candidate_text,candidate_type,similarity_score,positive
0,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,1.000000,Collaboration may involve implementing evidenc...
1,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.912461,Collaboration may involve implementing evidenc...
2,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.908266,Collaboration may involve implementing evidenc...
3,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.894374,Collaboration may involve implementing evidenc...
4,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.893983,Collaboration may involve implementing evidenc...


Removing the true positive from candidates

In [8]:
phase7_candidates["is_positive"] = (
    phase7_candidates["candidate_text"].str.strip().str.lower() ==
    phase7_candidates["positive"].str.strip().str.lower()
)

hard_negative_pool = phase7_candidates[phase7_candidates["is_positive"] == False].copy()

print("Hard negative pool shape:", hard_negative_pool.shape)
hard_negative_pool.head(20)

Hard negative pool shape: (25425, 7)


,query_id,query,candidate_text,candidate_type,similarity_score,positive,is_positive
0,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,1.000000,Collaboration may involve implementing evidenc...,False
1,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.912461,Collaboration may involve implementing evidenc...,False
2,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.908266,Collaboration may involve implementing evidenc...,False
3,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.894374,Collaboration may involve implementing evidenc...,False
4,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.893983,Collaboration may involve implementing evidenc...,False
5,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.891884,Collaboration may involve implementing evidenc...,False
6,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.887956,Collaboration may involve implementing evidenc...,False
7,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.884563,Collaboration may involve implementing evidenc...,False
8,0,How can blood donation centers collaborate wit...,How can blood donation centers partner with he...,question,0.882852,Collaboration may involve implementing evidenc...,False
9,0,How can blood donation centers collaborate wit...,How do blood donation centers collaborate with...,question,0.881042,Collaboration may involve implementing evidenc...,False


Keeping the top hard negatives per query

In [9]:
hard_negative_pool = hard_negative_pool.sort_values(
    ["query_id", "similarity_score"],
    ascending=[True, False]
)

top_hard_negatives = hard_negative_pool.groupby("query_id").head(3).reset_index(drop=True)

print("Top hard negatives shape:", top_hard_negatives.shape)
top_hard_negatives.head(20)

Top hard negatives shape: (8100, 7)


,query_id,query,candidate_text,candidate_type,similarity_score,positive,is_positive
0,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,1.000000,Collaboration may involve implementing evidenc...,False
1,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.912461,Collaboration may involve implementing evidenc...,False
2,0,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,0.908266,Collaboration may involve implementing evidenc...,False
3,1,What strategies can be employed to address the...,What strategies can be employed to address the...,question,1.000000,Strategies to address stigma and discriminatio...,False
4,1,What strategies can be employed to address the...,What measures can be implemented to address th...,question,0.935312,Strategies to address stigma and discriminatio...,False
5,1,What strategies can be employed to address the...,What strategies can be implemented to reduce t...,question,0.917318,Strategies to address stigma and discriminatio...,False
6,2,What role do community events or festivals pla...,What role do community events or festivals pla...,question,1.000000,Community events or festivals provide opportun...,False
7,2,What role do community events or festivals pla...,What role do community-based events and festiv...,question,0.958345,Community events or festivals provide opportun...,False
8,2,What role do community events or festivals pla...,"What role can community-based events, such as ...",question,0.913314,Community events or festivals provide opportun...,False
9,3,How can blood donation centers collaborate wit...,How can blood donation centers collaborate wit...,question,1.000000,Blood donation centers can partner with transp...,False


In [12]:
# First inspect columns
print(top_hard_negatives.columns.tolist())

['query_id', 'query_x', 'candidate_text', 'candidate_type', 'similarity_score', 'positive_x', 'is_positive', 'query_y', 'positive_y', 'topic_x', 'topic_y']


Creating training triples

In [13]:
top_hard_negatives = top_hard_negatives.merge(
    phase6_gold[["query_id", "topic"]],
    on="query_id",
    how="left"
)

phase7_triples = top_hard_negatives.rename(columns={
    "query_x": "anchor",
    "positive_x": "positive",
    "candidate_text": "hard_negative",
    "topic_x": "topic"
})[["query_id", "anchor", "positive", "hard_negative", "similarity_score", "topic"]]

print("Phase 7 triples shape:", phase7_triples.shape)
phase7_triples.head(20)

Phase 7 triples shape: (8100, 7)


,query_id,anchor,positive,hard_negative,similarity_score,topic,topic
0,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,1.000000,Blood Type,Blood Type
1,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,0.912461,Blood Type,Blood Type
2,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,0.908266,Blood Type,Blood Type
3,1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,What strategies can be employed to address the...,1.000000,Other,Other
4,1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,What measures can be implemented to address th...,0.935312,Other,Other
5,1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,What strategies can be implemented to reduce t...,0.917318,Other,Other
6,2,What role do community events or festivals pla...,Community events or festivals provide opportun...,What role do community events or festivals pla...,1.000000,Campaign/Event,Campaign/Event
7,2,What role do community events or festivals pla...,Community events or festivals provide opportun...,What role do community-based events and festiv...,0.958345,Campaign/Event,Campaign/Event
8,2,What role do community events or festivals pla...,Community events or festivals provide opportun...,"What role can community-based events, such as ...",0.913314,Campaign/Event,Campaign/Event
9,3,How can blood donation centers collaborate wit...,Blood donation centers can partner with transp...,How can blood donation centers collaborate wit...,1.000000,Blood Type,Blood Type


Checking for duplicates and invalid rows

In [14]:
phase7_triples = phase7_triples.drop_duplicates(
    subset=["anchor", "positive", "hard_negative"]
).reset_index(drop=True)

phase7_triples = phase7_triples[
    phase7_triples["anchor"].str.strip().ne("") &
    phase7_triples["positive"].str.strip().ne("") &
    phase7_triples["hard_negative"].str.strip().ne("")
].reset_index(drop=True)

print("Cleaned Phase 7 triples shape:", phase7_triples.shape)

Cleaned Phase 7 triples shape: (8100, 7)


Inspecting examples manually

In [15]:
phase7_triples.sample(15, random_state=42)

,query_id,anchor,positive,hard_negative,similarity_score,topic,topic
2818,939,How do donors ensure they meet the eligibility...,Donors can ensure they meet the eligibility cr...,How are donors screened for eligibility before...,0.926061,Eligibility,Eligibility
6922,2307,What role does donor education and awareness p...,Donor education and awareness play a crucial r...,What role does donor education play in promoti...,0.922599,Campaign/Event,Campaign/Event
5993,1997,What strategies can blood donation organizatio...,Strategies to engage with donors from diverse ...,What strategies can blood donation organizatio...,0.941568,Other,Other
3245,1081,Can you discuss the challenges and opportuniti...,"Challenges include digital divide, internet ac...",Can you discuss the potential impact of digita...,0.868595,Other,Other
7081,2360,What measures can be implemented to address th...,Measures may include targeted recruitment camp...,How do blood donation centers in Tanzania enga...,0.858848,Other,Other
554,184,How do blood donation centers encourage regula...,Blood donation centers encourage regular blood...,How do blood donation centers implement strate...,0.909607,Donation Center,Donation Center
2646,882,How do blood donation organizations collaborat...,Blood donation organizations collaborate with ...,How do blood donation organizations collaborat...,1.000000,Blood Type,Blood Type
4240,1413,How can healthcare policies support blood dona...,Healthcare policies can support blood donation...,How can healthcare policies support voluntary ...,0.957866,Other,Other
6847,2282,How do automated donor recruitment and engagem...,Automated donor recruitment and engagement too...,How do automated donor recruitment and engagem...,0.962742,Other,Other
6143,2047,How can blood donation facilitation programs a...,Blood donation facilitation programs can adapt...,What measures can blood donation facilitation ...,0.885554,Blood Type,Blood Type


Topic distribution of hard negatives

In [17]:
print(phase7_triples.columns.tolist())

['query_id', 'anchor', 'positive', 'hard_negative', 'similarity_score', 'topic', 'topic']


In [18]:
# First inspect which topic column has the correct values
print(top_hard_negatives[["topic_x", "topic_y"]].head(10))

          topic_x         topic_y
0      Blood Type      Blood Type
1      Blood Type      Blood Type
2      Blood Type      Blood Type
3           Other           Other
4           Other           Other
5           Other           Other
6  Campaign/Event  Campaign/Event
7  Campaign/Event  Campaign/Event
8  Campaign/Event  Campaign/Event
9      Blood Type      Blood Type


In [22]:
phase7_triples = phase7_triples.loc[:, ~phase7_triples.columns.duplicated()]
print(phase7_triples.columns.tolist())

['query_id', 'anchor', 'positive', 'hard_negative', 'similarity_score', 'topic']


In [19]:
phase7_triples = pd.DataFrame({
    "query_id": top_hard_negatives["query_id"],
    "anchor": top_hard_negatives["query_x"],
    "positive": top_hard_negatives["positive_x"],
    "hard_negative": top_hard_negatives["candidate_text"],
    "similarity_score": top_hard_negatives["similarity_score"],
    "topic": top_hard_negatives["topic_x"]
})

print(phase7_triples.columns.tolist())
phase7_triples.head()

['query_id', 'anchor', 'positive', 'hard_negative', 'similarity_score', 'topic']


,query_id,anchor,positive,hard_negative,similarity_score,topic
0,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,1.000000,Blood Type
1,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,0.912461,Blood Type
2,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,0.908266,Blood Type
3,1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,What strategies can be employed to address the...,1.000000,Other
4,1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,What measures can be implemented to address th...,0.935312,Other


In [20]:
phase7_triples = pd.DataFrame({
    "query_id": top_hard_negatives["query_id"],
    "anchor": top_hard_negatives["query_x"],
    "positive": top_hard_negatives["positive_x"],
    "hard_negative": top_hard_negatives["candidate_text"],
    "similarity_score": top_hard_negatives["similarity_score"],
    "topic": top_hard_negatives["topic_y"]
})

print(phase7_triples.columns.tolist())
phase7_triples.head()

['query_id', 'anchor', 'positive', 'hard_negative', 'similarity_score', 'topic']


,query_id,anchor,positive,hard_negative,similarity_score,topic
0,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,1.000000,Blood Type
1,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,0.912461,Blood Type
2,0,How can blood donation centers collaborate wit...,Collaboration may involve implementing evidenc...,How can blood donation centers collaborate wit...,0.908266,Blood Type
3,1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,What strategies can be employed to address the...,1.000000,Other
4,1,What strategies can be employed to address the...,Strategies to address stigma and discriminatio...,What measures can be implemented to address th...,0.935312,Other


In [21]:
print(phase7_triples.columns.tolist())

['query_id', 'anchor', 'positive', 'hard_negative', 'similarity_score', 'topic']


In [23]:
print(phase7_triples["topic"].value_counts())

topic
Other              3201
Blood Type         1902
Donation Center    1320
Campaign/Event      663
Disease/Illness     549
Eligibility         150
Medication          114
Deferral Period     102
Donor History        99
Name: count, dtype: int64


Similarity summary of hard negatives

In [24]:
print(phase7_triples["similarity_score"].describe())

count    8100.000000
mean        0.929908
std         0.069057
min         0.362517
25%         0.883992
50%         0.936381
75%         1.000000
max         1.000000
Name: similarity_score, dtype: float64


Saving Phase 7 outputs

In [25]:
phase7_triples.to_csv("phase7_training_triples.csv", index=False)
top_hard_negatives.to_csv("phase7_top_hard_negatives.csv", index=False)
hard_negative_pool.to_csv("phase7_full_hard_negative_pool.csv", index=False)

print("Saved Phase 7 files:")
print("- phase7_training_triples.csv")
print("- phase7_top_hard_negatives.csv")
print("- phase7_full_hard_negative_pool.csv")

Saved Phase 7 files:
- phase7_training_triples.csv
- phase7_top_hard_negatives.csv
- phase7_full_hard_negative_pool.csv


Creating Phase 7 summary table

In [26]:
phase7_summary = {
    "Queries used": len(query_texts),
    "Candidate matches mined": len(candidate_matches_df),
    "Hard negative pool size": len(hard_negative_pool),
    "Final training triples": len(phase7_triples),
    "Average hard negative similarity": round(phase7_triples["similarity_score"].mean(), 4)
}

phase7_summary_df = pd.DataFrame(list(phase7_summary.items()), columns=["Metric", "Value"])
phase7_summary_df

,Metric,Value
0,Queries used,2700.0000
1,Candidate matches mined,27000.0000
2,Hard negative pool size,25425.0000
3,Final training triples,8100.0000
4,Average hard negative similarity,0.9299
